## Setup and derived features
squad_value_diff and form_diff aren't new information — they're linear combinations of columns already in the table — but giving a tree model an explicit "advantage" feature directly usually lets it find a good split in fewer levels than forcing it to compare two separate columns implicitly.

In [0]:
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer, VectorAssembler, VectorIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

CATALOG = "workspace"
SCHEMA = "transfermarkt"

data = spark.read.parquet(f"/Volumes/{CATALOG}/{SCHEMA}/ml/classification_features_final")

data = (data
    .withColumn("squad_value_diff", F.col("home_squad_value_eur") - F.col("away_squad_value_eur"))
    .withColumn("form_diff", F.col("home_rolling_win_rate") - F.col("away_rolling_win_rate")))

data.groupBy("year").count().orderBy("year").show()

## Time-based split, not a random one
This is forecasting a future result from past information, not i.i.d. classification — a random 80/20 split would let the model train on 2023 matches and get tested on 2013 ones, which is never how it would actually be used and can flatter performance if there's any drift over time. Train on the oldest years, tune on the middle years, touch the most recent year exactly once for final evaluation.

In [0]:
train = data.filter(F.col("year") <= 2020)
val = data.filter((F.col("year") >= 2021) & (F.col("year") <= 2022))
test = data.filter(F.col("year") == 2023)

print("train:", train.count(), " val:", val.count(), " test:", test.count())

## The real baseline
The bar to beat is this scoped dataset's own majority-class rate on the actual test set — not the 45.1% population figure from the original EDA, for the reason discussed above.

In [0]:
majority_class = train.groupBy("result").count().orderBy(F.desc("count")).first()["result"]
test_majority_acc = test.filter(F.col("result") == majority_class).count() / test.count()

print("Majority class:", majority_class)
print(f"Majority-class baseline accuracy on test: {test_majority_acc:.3f}")
test.groupBy("result").count().withColumn("pct", F.round(F.col("count") / test.count() * 100, 1)).show()

## Feature preparation
competition_key is a surrogate key, not a continuous quantity — feeding it in raw would wrongly imply competition #40 is "greater than" competition #12. VectorIndexer inspects each assembled column and auto-flags any with fewer than maxCategories distinct values as categorical (competition_key has ~65 — well under 100), so it gets one-hot-style tree splits instead of being treated as ordinal. Both this and the label indexer are fit on the full dataset rather than train-only — that's safe, not leakage, because neither one ever looks at the label; they only need to see every category value that exists so val/test don't crash on a category train never saw.

In [0]:
label_indexer = StringIndexer(inputCol="result", outputCol="label")
label_model = label_indexer.fit(data)
print("Label mapping:", list(enumerate(label_model.labels)))

feature_cols = ["home_rolling_win_rate", "away_rolling_win_rate",
                "home_squad_value_eur", "away_squad_value_eur",
                "squad_value_diff", "form_diff", "competition_key"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")
indexer = VectorIndexer(inputCol="raw_features", outputCol="features", maxCategories=100)

vi_model = indexer.fit(assembler.transform(label_model.transform(data)))

def prep(df):
    return vi_model.transform(assembler.transform(label_model.transform(df)))

train_prepped = prep(train)
val_prepped = prep(val)

## Hyperparameter search on the validation set
A small manual grid, not Spark's built-in CrossValidator — CrossValidator does random k-fold splits by default, which would reintroduce the exact temporal leakage the time-based split was built to avoid. Selecting by weighted F1 rather than accuracy, since the classes aren't balanced.

In [0]:
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

best_f1, best_params = -1, None
for num_trees in [50, 100]:
    for max_depth in [5, 10]:
        rf = RandomForestClassifier(labelCol="label", featuresCol="features",
                                     numTrees=num_trees, maxDepth=max_depth, seed=42)
        model = rf.fit(train_prepped)
        f1 = evaluator_f1.evaluate(model.transform(val_prepped))
        print(f"numTrees={num_trees}, maxDepth={max_depth}: val F1={f1:.4f}")
        if f1 > best_f1:
            best_f1, best_params = f1, (num_trees, max_depth)

print("Best params:", best_params, " val F1:", best_f1)

## Final model — train on train+val, touch test exactly once
Retrains with the winning hyperparameters on the combined train+validation data, then evaluates on test a single time. Reporting the confusion matrix alongside accuracy, since a good overall number can still hide a model that never correctly predicts draws — a well-known hard class in football outcome prediction.

In [0]:
train_val_prepped = prep(train.union(val))
test_prepped = prep(test)

final_rf = RandomForestClassifier(labelCol="label", featuresCol="features",
                                   numTrees=best_params[0], maxDepth=best_params[1], seed=42)
final_model = final_rf.fit(train_val_prepped)
test_preds = final_model.transform(test_prepped)

evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
print("Test accuracy:", evaluator_acc.evaluate(test_preds))
print("Test weighted F1:", evaluator_f1.evaluate(test_preds))
print("Majority-class baseline accuracy:", test_majority_acc)

test_preds.groupBy("label", "prediction").count().orderBy("label", "prediction").show(20)

## Feature importances

In [0]:
for name, score in sorted(zip(feature_cols, final_model.featureImportances.toArray()), key=lambda x: -x[1]):
    print(f"{name}: {score:.4f}")